# BERT + LoRA: comparing author styles

Fine-tune a shared BERT encoder on **same-author / different-author** paper pairs. Only titles and abstracts enter the model; first-author ORCIDs supply labels.

Run the cells in order. Install `3_train/requirements.txt` in the notebook's Python environment first. A CUDA GPU is recommended. This learns authorship similarity, which may include topic as well as style; it is not a topic-controlled style benchmark.

## 1. Imports
Keep dependencies here and experiment settings in the next section.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import (
    LoraConfig, TaskType, get_peft_model,
    get_peft_model_state_dict, set_peft_model_state_dict,
)
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, set_seed


## 2. Configuration
Defaults use `sciar_refereed_v1` (at least two papers per author). Pair counts are sampled examples, not all possible pairs. For a smoke run, set pair counts to `64`, `32`, `32` and epochs to `1`.

In [ ]:
# Works when Jupyter starts from the repository root or 3_train/.
ROOT = Path.cwd().resolve()
if ROOT.name == '3_train':
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/processed/sciar_refereed_v1/papers.parquet'
OUTPUT_DIR = ROOT / '3_train/outputs/bert_lora_v1'

MODEL_NAME = 'bert-base-uncased'
SEED = 42
TRAIN_FRACTION = 0.8
VAL_FRACTION = 0.1  # Remaining authors are held out for test.
TRAIN_PAIRS = 20_000
VAL_PAIRS = 2_000
TEST_PAIRS = 2_000
MAX_LENGTH = 256  # Per paper, including title and abstract.
BATCH_SIZE = 8  # Each pair contains two independently encoded papers.
EPOCHS = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MARGIN = 0.2
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DATA_PATH.is_file(), f'Dataset not found: {DATA_PATH}'
assert 0 < TRAIN_FRACTION < 1 and 0 < VAL_FRACTION < 1 - TRAIN_FRACTION
assert all(n >= 2 and n % 2 == 0 for n in (TRAIN_PAIRS, VAL_PAIRS, TEST_PAIRS))
assert EPOCHS >= 1 and BATCH_SIZE >= 1 and 8 <= MAX_LENGTH <= 512
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(f'Device: {DEVICE} | Output: {OUTPUT_DIR}')


## 3. Load papers and split authors
Assign authors to **80% train / 10% validation / 10% test** before generating pairs. No author or paper crosses a split. Authors with fewer than two papers are excluded. Save the exact author assignments for reproducibility.

In [ ]:
papers = pd.read_parquet(DATA_PATH, columns=['bibcode', 'first_author_orcid', 'title', 'abstract'])
assert papers.notna().all().all(), 'Required paper fields contain missing values.'
assert papers['bibcode'].is_unique, 'Expected one row per bibcode.'
assert papers[['title', 'abstract']].map(lambda text: isinstance(text, str) and bool(text.strip())).all().all()
counts = papers.groupby('first_author_orcid')['bibcode'].transform('size')
papers = papers.loc[counts >= 2].reset_index(drop=True)
authors = np.array(sorted(papers['first_author_orcid'].unique()))
np.random.default_rng(SEED).shuffle(authors)
train_end = int(len(authors) * TRAIN_FRACTION)
val_end = train_end + int(len(authors) * VAL_FRACTION)
author_splits = {
    'train': authors[:train_end].tolist(),
    'validation': authors[train_end:val_end].tolist(),
    'test': authors[val_end:].tolist(),
}
assert all(len(ids) >= 2 for ids in author_splits.values()), 'Need at least two authors per split.'
paper_groups = papers.groupby('first_author_orcid').indices
(OUTPUT_DIR / 'author_splits.json').write_text(json.dumps(author_splits, indent=2))
for name, ids in author_splits.items():
    print(f'{name}: {len(ids):,} authors, {sum(len(paper_groups[a]) for a in ids):,} papers')


## 4. Sample balanced pairs
Half the pairs contain distinct papers by one author (`+1`); half contain papers by different authors (`-1`). Authors are sampled uniformly so prolific authors do not dominate. Sampling is with replacement across pairs, so a paper or pair can recur within its split. Validation and test pairs stay fixed.

In [ ]:
def sample_pairs(author_ids, n_pairs, seed):
    rng = random.Random(seed)
    pairs = []
    for _ in range(n_pairs // 2):
        author = rng.choice(author_ids)
        left, right = rng.sample(list(paper_groups[author]), 2)
        pairs.append((int(left), int(right), 1))
        author_a, author_b = rng.sample(author_ids, 2)
        left = rng.choice(paper_groups[author_a])
        right = rng.choice(paper_groups[author_b])
        pairs.append((int(left), int(right), -1))
    rng.shuffle(pairs)
    return pairs

pair_sets = {
    name: sample_pairs(author_splits[name], size, SEED + offset)
    for offset, (name, size) in enumerate(
        [('train', TRAIN_PAIRS), ('validation', VAL_PAIRS), ('test', TEST_PAIRS)]
    )
}
for name, pairs in pair_sets.items():
    pd.DataFrame(pairs, columns=['left_row', 'right_row', 'label']).assign(
        left_bibcode=[papers.at[i, 'bibcode'] for i, _, _ in pairs],
        right_bibcode=[papers.at[j, 'bibcode'] for _, j, _ in pairs],
    ).to_csv(OUTPUT_DIR / f'{name}_pairs.csv', index=False)
    print(f'{name}: {len(pairs):,} pairs')


## 5. Tokenizer and LoRA encoder
Tokenize each paper as `[CLS] title [SEP] abstract [SEP]`. Both papers share the same encoder. Only LoRA query/value adapters train; the base BERT weights stay frozen. Masked mean pooling produces a normalized embedding without using BERT's pooler.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False)
model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['query', 'value'], bias='none',
)).to(DEVICE)
model.print_trainable_parameters()

def collate_pairs(batch):
    # First half of the batch is left papers; second half is right papers.
    indices = [i for i, _, _ in batch] + [j for _, j, _ in batch]
    rows = papers.iloc[indices]
    tokens = tokenizer(
        rows['title'].tolist(), rows['abstract'].tolist(),
        padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt',
    )
    labels = torch.tensor([label for _, _, label in batch], dtype=torch.float32)
    return tokens, labels

loaders = {
    name: DataLoader(
        pairs, batch_size=BATCH_SIZE, shuffle=(name == 'train'),
        collate_fn=collate_pairs, num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )
    for name, pairs in pair_sets.items()
}

def encode(tokens):
    tokens = {key: value.to(DEVICE) for key, value in tokens.items()}
    hidden = model(**tokens).last_hidden_state
    mask = tokens['attention_mask'].unsqueeze(-1).to(hidden.dtype)
    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return F.normalize(pooled, p=2, dim=-1)


## 6. Loss and evaluation
Use `CosineEmbeddingLoss(margin=0.2)`: **same author:** `1 − cosine_similarity`; **different author:** `max(0, cosine_similarity − margin)`. Average over the batch. Validation ROC-AUC selects the best epoch; a validation-only threshold maximizes balanced accuracy (equivalent to accuracy for these balanced pairs). Test data never selects an epoch or threshold.

In [ ]:
criterion = torch.nn.CosineEmbeddingLoss(margin=MARGIN)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    for tokens, labels in tqdm(loader, desc='Evaluate', leave=False):
        left, right = encode(tokens).chunk(2)
        loss = criterion(left, right, labels.to(DEVICE))
        total_loss += loss.item() * len(labels)
        all_labels.extend((labels > 0).int().tolist())
        all_scores.extend((left * right).sum(dim=-1).cpu().tolist())
    return total_loss / len(loader.dataset), np.array(all_labels), np.array(all_scores)


## 7. Train
Record an untrained validation baseline, then optimize the LoRA parameters. Keep the best adapter in memory according to validation ROC-AUC. Training uses full precision for simplicity.

In [ ]:
baseline_loss, baseline_labels, baseline_scores = evaluate(loaders['validation'])
baseline_auc = float(roc_auc_score(baseline_labels, baseline_scores))
print(f'Before training: validation loss={baseline_loss:.4f}, AUC={baseline_auc:.4f}')
history, best_auc, best_state = [], -float('inf'), None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for tokens, labels in tqdm(loaders['train'], desc=f'Epoch {epoch}/{EPOCHS}'):
        optimizer.zero_grad(set_to_none=True)
        left, right = encode(tokens).chunk(2)
        loss = criterion(left, right, labels.to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
    val_loss, val_labels, val_scores = evaluate(loaders['validation'])
    val_auc = float(roc_auc_score(val_labels, val_scores))
    row = dict(epoch=epoch, train_loss=total_loss / TRAIN_PAIRS, val_loss=val_loss, val_auc=val_auc)
    history.append(row)
    print(row)
    if val_auc > best_auc:
        best_auc, best_epoch = val_auc, epoch
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in get_peft_model_state_dict(model).items()
        }
set_peft_model_state_dict(model, best_state)
pd.DataFrame(history)


## 8. Held-out evaluation and export
Choose the decision threshold using the best adapter's validation scores, then evaluate the test set once. Save the adapter, tokenizer, metrics, configuration, history, and sampling manifests. The saved model needs the original base BERT checkpoint; it is not a standalone merged model.

In [ ]:
val_loss, val_labels, val_scores = evaluate(loaders['validation'])
fpr, tpr, thresholds = roc_curve(val_labels, val_scores)
finite = np.isfinite(thresholds)
threshold = float(thresholds[finite][np.argmax((tpr - fpr)[finite])])
test_loss, test_labels, test_scores = evaluate(loaders['test'])
metrics = {
    'baseline_val_auc': baseline_auc, 'best_epoch': best_epoch,
    'val_auc': float(roc_auc_score(val_labels, val_scores)),
    'val_accuracy': float(accuracy_score(val_labels, val_scores >= threshold)),
    'threshold': threshold, 'test_loss': test_loss,
    'test_auc': float(roc_auc_score(test_labels, test_scores)),
    'test_accuracy': float(accuracy_score(test_labels, test_scores >= threshold)),
}
model.save_pretrained(OUTPUT_DIR / 'adapter')
tokenizer.save_pretrained(OUTPUT_DIR / 'adapter')
config = {key: str(value) if isinstance(value, Path) else value
          for key, value in list(globals().items())
          if key.isupper() and isinstance(value, (str, int, float, Path))}
config.update(pooling='attention-mask mean, including special tokens', normalize_embeddings=True)
(OUTPUT_DIR / 'config.json').write_text(json.dumps(config, indent=2))
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'history.csv', index=False)
print(json.dumps(metrics, indent=2))
print(f'Saved to {OUTPUT_DIR}')
